<h1 class="alert alert-block alert-info">Different Vector Similarity Algorithms</h1>
<div class="alert alert-block alert-info">

1. Cosine similarity.
   
2. Dot product.
   
3. Euclidean (L2) distance</div>

<div class="alert alert-block">
<h2 class="alert alert-info">Cosine Similarity</h2>
<div class="alert alert-info">

1. Calcuate the cosine similarity which is measuring the angle between two vectors (ignoring their magnitude/length).
2. State of the art models uses it during training.
1. OpenAI / AzureOpenAI
   <span style="color:orange;">
   Vector embedding provided by AzureOpenAI is normalized.  
    </span>
</div>

<div class="alert alert-danger">

1. Its computational heavy if vectors aren't normalized
</div>

<div class="alert alert-success">

1. Its robust against the variation in text length.
2. It focuses on orientation.
3. Value range: [0, 1]. 
   1. Higher is better. 
   2. 1 -> more identical.
4. It can handle the different length of the documents.
</div>

In [ ]:
from langchain_openai import AzureChatOpenAI, ChatOpenAI, AzureOpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

endpoint = os.getenv("AZURE_ENDPOINT")
model = os.getenv("AZURE_OPENAI_EMBEDDING_LARGE_MODEL")

subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

openai_embedding_llm = AzureOpenAIEmbeddings( 
        api_version=api_version,
        azure_deployment=model,
   )

<div class="alert alert-block alert-info">We can apply dot-product here because <span style="color:red;">vector embeddings are unit normalized.</span></div>

In [ ]:
import numpy as np
pairs = [("I love AI", "I hate AI"),
          ("I hate AI", "I hate Artificial Intelligence."),
          ("love", "hate")]
for (a, b) in pairs:
    embed_first, embed_second = openai_embedding_llm.embed_documents([a,b])
    vector_a = np.array(embed_first)
    vector_b = np.array(embed_second)

    dot_product_similarity = np.dot(vector_a, vector_b)
    print(f"Cosine Similarity {a} & {b} =>  {dot_product_similarity}")

In [40]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

docs = [
    "I love AI",
    "I hate AI",
    "I hate Artificial Intelligence.",
    "love",
    "hate",
    "I have dog",
    "Dog barkes"]


<div class="alert alert-block alert-info">FAISS doesn't support Cosine similarity by default. 

It leverages the concept that <span style="color:green;">Cosine similarity = Dot product if vectors are normalized ( = 1)</span> </br>
<span style="color:orange;">Langchain implemented it by doing following:</span> 

1. Normalized vector embedding
2. Perform dot product (or Inner product)

<span style="color:red;">cos(theta) = query * vector 
<br>iff query and vector is normalized
<br>Otherwise, cos(theta) = (query * vector) / (|query| * |vector|)</span>

</div>



<div class="alert alert-block alert-success">
Similarity Score -> Lower the better. </div>
<div class="alert alert-block alert-success">
Similarity Relevance Score -> Higher the better. </div>

In [51]:
vector_db = FAISS.from_texts(docs, openai_embedding_llm, distance_strategy=DistanceStrategy.COSINE)

for doc in docs:
    similarity = vector_db.similarity_search_with_score(query=doc, k=3)
    print(f"Compare with top 3 results for query: {doc}")
    for sim in similarity:
        print(f"Query: {doc} -> Similairity search with {sim[0].page_content} -> {sim[1]}")
        # print(sim)

Compare with top 3 results for query: I love AI
Query: I love AI -> Similairity search with I love AI -> 1.0252385891362792e-06
Query: I love AI -> Similairity search with I hate AI -> 0.5083215236663818
Query: I love AI -> Similairity search with I hate Artificial Intelligence. -> 0.8266680240631104
Compare with top 3 results for query: I hate AI
Query: I hate AI -> Similairity search with I hate AI -> 1.252609081348055e-06
Query: I hate AI -> Similairity search with I hate Artificial Intelligence. -> 0.3754262924194336
Query: I hate AI -> Similairity search with I love AI -> 0.5083571672439575
Compare with top 3 results for query: I hate Artificial Intelligence.
Query: I hate Artificial Intelligence. -> Similairity search with I hate Artificial Intelligence. -> 8.993207529783831e-07
Query: I hate Artificial Intelligence. -> Similairity search with I hate AI -> 0.3749431371688843
Query: I hate Artificial Intelligence. -> Similairity search with I love AI -> 0.8264491558074951
Compare 

<div class="alert alert-block alert-info">MIP (Max Inner Product). 

It leverages the concept that <span style="color:green;">
1. Cosine similarity = Max IP
2. Range (-inf , inf)
3. Don't need normalization</span> </br>

<span style="color:red;">cos(theta) = query * vector 

In [48]:
vector_db = FAISS.from_texts(docs, openai_embedding_llm, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc in docs:
    similarity = vector_db.similarity_search_with_score(query=doc, k=3)
    print(f"Compare with top 3 results")
    for sim in similarity:
        print(f"Query: {doc} -> Similairity search with {sim[0].page_content} -> {sim[1]}")

Compare with top 3 results
Query: I love AI -> Similairity search with I love AI -> 1.0003247261047363
Query: I love AI -> Similairity search with I hate AI -> 0.7458242774009705
Query: I love AI -> Similairity search with I hate Artificial Intelligence. -> 0.5868222117424011
Compare with top 3 results
Query: I hate AI -> Similairity search with I hate AI -> 0.9999535083770752
Query: I hate AI -> Similairity search with I hate Artificial Intelligence. -> 0.8123337626457214
Query: I hate AI -> Similairity search with I love AI -> 0.7461401224136353
Compare with top 3 results
Query: I hate Artificial Intelligence. -> Similairity search with I hate Artificial Intelligence. -> 0.9997939467430115
Query: I hate Artificial Intelligence. -> Similairity search with I hate AI -> 0.8122041821479797
Query: I hate Artificial Intelligence. -> Similairity search with I love AI -> 0.5869070291519165
Compare with top 3 results
Query: love -> Similairity search with love -> 0.9997146725654602
Query: lov

<div class="alert alert-block alert-info">Euclidean distance (L2)


<span style="color:green;">

1. L2
   1. Shortest distance between 2 points in 2D space.
   2. Shortest distance between N points in ND space. N=1536
2. Range (0 , inf)
3. Doesn't needs normalization.
4. Lower is better.
5. Higher Cosine similarity = Lower L2</span> </br>

<span style="color:red;">

1. L2 cares if vectors are in different length/magnitude. </span>
</div>

In [50]:
vector_db = FAISS.from_texts(docs, openai_embedding_llm, distance_strategy=DistanceStrategy.EUCLIDEAN_DISTANCE)

for doc in docs:
    similarity = vector_db.similarity_search_with_score(query=doc, k=3)
    print(f"Compare with top 3 results")
    for sim in similarity:
        print(f"Query: {doc} -> Similairity search with {sim[0].page_content} -> {sim[1]}")

Compare with top 3 results
Query: I love AI -> Similairity search with I love AI -> 0.0
Query: I love AI -> Similairity search with I hate AI -> 0.5083297491073608
Query: I love AI -> Similairity search with I hate Artificial Intelligence. -> 0.8264729380607605
Compare with top 3 results
Query: I hate AI -> Similairity search with I hate AI -> 0.0
Query: I hate AI -> Similairity search with I hate Artificial Intelligence. -> 0.3752489686012268
Query: I hate AI -> Similairity search with I love AI -> 0.5083297491073608
Compare with top 3 results
Query: I hate Artificial Intelligence. -> Similairity search with I hate Artificial Intelligence. -> 0.0
Query: I hate Artificial Intelligence. -> Similairity search with I hate AI -> 0.3752489686012268
Query: I hate Artificial Intelligence. -> Similairity search with I love AI -> 0.8264729380607605
Compare with top 3 results
Query: love -> Similairity search with love -> 1.4971016071285703e-06
Query: love -> Similairity search with hate -> 0.99